# Prophet proof of concept: daily airport delay rate

This notebook tests whether Prophet adds value for an **aggregated** forecasting problem: the daily percentage of arrivals delayed by more than 15 minutes at one high-volume destination airport. It does not replace the flight-level T-60 Ridge model.

Design safeguards:

- The airport is selected using the training partition only.
- A daily observation is retained only when `flight_count >= 20`.
- Yearly seasonality is disabled because the available monthly snapshots do not form several complete years.
- December 2022 is used for model development and validation. March and June 2023 remain locked unless explicitly enabled at the end.
- Prophet is compared with simple baselines; it is useful only if it improves out-of-time error.


## 1. Environment

Prophet is intentionally not installed automatically. If the import below fails, run `%pip install prophet` in a separate notebook cell, restart the kernel, and rerun the notebook. Once the experiment is accepted, add Prophet to the project's locked dependencies. On Windows, the setup cell places CmdStan's bundled `tbb.dll` beside its executable when needed; this avoids accidentally loading an incompatible system-wide TBB runtime.

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as pads
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    import prophet
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Prophet is not installed. Run `%pip install prophet`, restart the kernel, "
        "and rerun this notebook."
    ) from exc


def prepare_windows_prophet_runtime() -> None:
    """Place CmdStan's bundled TBB beside its executable on Windows."""
    if sys.platform != "win32":
        return
    stan_model_dir = Path(prophet.__file__).resolve().parent / "stan_model"
    target = stan_model_dir / "tbb.dll"
    if target.exists():
        return
    candidates = list(
        stan_model_dir.glob("cmdstan-*/stan/lib/stan_math/lib/tbb/tbb.dll")
    )
    if len(candidates) != 1:
        raise RuntimeError(
            "Could not identify the TBB runtime bundled with Prophet's CmdStan."
        )
    shutil.copy2(candidates[0], target)


prepare_windows_prophet_runtime()
Prophet = prophet.Prophet

pd.set_option("display.max_columns", 30)
RANDOM_SEED = 42


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "README.md").exists() and (directory / "data").exists():
            return directory
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "data" / "processed" / "expanded_arrival_pre_t60"
PARTITIONS = {
    "train": DATASET_ROOT / "train",
    "validation": DATASET_ROOT / "validation",
    "test": DATASET_ROOT / "test",
    "future_test": DATASET_ROOT / "future_test",
}

MIN_DAILY_FLIGHTS = 20
AIRPORT_OVERRIDE: str | None = None  # Example: "EHAM"; None selects the busiest training airport.

missing = [name for name, path in PARTITIONS.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected Parquet partitions: {missing}")

PROJECT_ROOT, DATASET_ROOT


## 2. Select a high-volume airport without looking at validation or test

The destination airport is chosen exclusively from valid training rows. This prevents future traffic from influencing the experiment definition.

In [ ]:
def parquet_dataset(path: Path) -> pads.Dataset:
    return pads.dataset(path, format="parquet")


def select_busiest_destination(train_path: Path) -> tuple[str, pd.DataFrame]:
    table = parquet_dataset(train_path).to_table(columns=["ADES", "Arrival_Delay_Min"])
    frame = table.to_pandas()
    frame = frame.dropna(subset=["ADES", "Arrival_Delay_Min"])
    ranking = (
        frame.groupby("ADES", observed=True)
        .size()
        .rename("training_flights")
        .sort_values(ascending=False)
        .reset_index()
    )
    if ranking.empty:
        raise ValueError("No valid destination airports were found in training.")
    return str(ranking.iloc[0]["ADES"]), ranking


busiest_airport, airport_ranking = select_busiest_destination(PARTITIONS["train"])
selected_airport = AIRPORT_OVERRIDE or busiest_airport
print(f"Selected destination airport: {selected_airport}")
airport_ranking.head(10)


## 3. Build the daily time series

`y` is the percentage of arrivals whose actual arrival was more than 15 minutes later than the filed arrival. `flight_count` is retained for volume checks and weighted evaluation, but it is not used as a Prophet regressor in this first experiment.

In [ ]:
REQUIRED_COLUMNS = ["ADES", "FILED ARRIVAL TIME", "Arrival_Delay_Min"]


def build_daily_airport_series(
    partition_path: Path,
    airport: str,
    split_name: str,
    min_daily_flights: int = MIN_DAILY_FLIGHTS,
) -> pd.DataFrame:
    dataset = parquet_dataset(partition_path)
    table = dataset.to_table(
        columns=REQUIRED_COLUMNS,
        filter=pads.field("ADES") == airport,
    )
    frame = table.to_pandas().dropna(subset=REQUIRED_COLUMNS).copy()
    frame["ds"] = pd.to_datetime(frame["FILED ARRIVAL TIME"]).dt.normalize()
    frame["delayed_over_15"] = frame["Arrival_Delay_Min"].gt(15)

    daily = (
        frame.groupby("ds", as_index=False)
        .agg(
            flight_count=("Arrival_Delay_Min", "size"),
            delayed_count=("delayed_over_15", "sum"),
            median_arrival_delay_min=("Arrival_Delay_Min", "median"),
        )
        .sort_values("ds")
    )
    daily = daily.loc[daily["flight_count"] >= min_daily_flights].copy()
    daily["y"] = 100.0 * daily["delayed_count"] / daily["flight_count"]
    daily["split"] = split_name
    return daily.reset_index(drop=True)


train_daily = build_daily_airport_series(PARTITIONS["train"], selected_airport, "train")
validation_daily = build_daily_airport_series(
    PARTITIONS["validation"], selected_airport, "validation"
)

series_summary = pd.concat([train_daily, validation_daily], ignore_index=True)
series_summary.groupby("split").agg(
    first_day=("ds", "min"),
    last_day=("ds", "max"),
    observed_days=("ds", "size"),
    flights=("flight_count", "sum"),
    mean_delay_rate_pct=("y", "mean"),
)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for split_name, split_frame in series_summary.groupby("split", sort=False):
    ax.plot(split_frame["ds"], split_frame["y"], marker="o", label=split_name)
ax.set(
    title=f"Daily arrivals delayed >15 min · {selected_airport}",
    xlabel="Observed day",
    ylabel="Delayed arrivals (%)",
)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()


## 4. Fit Prophet and explicit baselines

The model uses weekly seasonality only. Predictions are requested directly for the validation dates instead of constructing an assumed continuous future calendar. Forecasts are clipped to the valid percentage range for scoring.

In [ ]:
def fit_prophet(train: pd.DataFrame) -> Prophet:
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=False,
        seasonality_mode="additive",
        changepoint_prior_scale=0.05,
        interval_width=0.80,
    )
    return model.fit(train[["ds", "y"]].copy())


def add_baselines(train: pd.DataFrame, validation: pd.DataFrame) -> pd.DataFrame:
    result = validation.copy()
    global_mean = float(train["y"].mean())
    weekday_means = train.assign(weekday=train["ds"].dt.dayofweek).groupby("weekday")["y"].mean()
    result["global_mean_pred"] = global_mean
    result["weekday_mean_pred"] = (
        result["ds"].dt.dayofweek.map(weekday_means).fillna(global_mean)
    )
    return result


prophet_model = fit_prophet(train_daily)
prophet_forecast = prophet_model.predict(validation_daily[["ds"]].copy())
validation_results = add_baselines(train_daily, validation_daily)
validation_results = validation_results.merge(
    prophet_forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]],
    on="ds",
    how="left",
    validate="one_to_one",
)
validation_results["prophet_pred"] = validation_results["yhat"].clip(0, 100)
validation_results.head()


## 5. Out-of-time evaluation

Alongside MAE and RMSE, weighted MAE gives more influence to days containing more flights. A useful result must beat both simple baselines, not merely produce an attractive curve.

In [ ]:
def weighted_mae(y_true: pd.Series, y_pred: pd.Series, weights: pd.Series) -> float:
    return float(np.average(np.abs(y_true - y_pred), weights=weights))


def score_predictions(frame: pd.DataFrame, prediction_columns: dict[str, str]) -> pd.DataFrame:
    rows = []
    for model_name, prediction_column in prediction_columns.items():
        rows.append(
            {
                "model": model_name,
                "MAE_pp": mean_absolute_error(frame["y"], frame[prediction_column]),
                "RMSE_pp": mean_squared_error(
                    frame["y"], frame[prediction_column]
                ) ** 0.5,
                "weighted_MAE_pp": weighted_mae(
                    frame["y"], frame[prediction_column], frame["flight_count"]
                ),
            }
        )
    return pd.DataFrame(rows).sort_values("MAE_pp").reset_index(drop=True)


validation_scores = score_predictions(
    validation_results,
    {
        "Prophet weekly": "prophet_pred",
        "Training weekday mean": "weekday_mean_pred",
        "Training global mean": "global_mean_pred",
    },
)
validation_scores.style.format(
    {"MAE_pp": "{:.2f}", "RMSE_pp": "{:.2f}", "weighted_MAE_pp": "{:.2f}"}
)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(validation_results["ds"], validation_results["y"], marker="o", label="Observed")
ax.plot(
    validation_results["ds"],
    validation_results["prophet_pred"],
    marker="o",
    label="Prophet",
)
ax.fill_between(
    validation_results["ds"],
    validation_results["yhat_lower"].clip(0, 100),
    validation_results["yhat_upper"].clip(0, 100),
    alpha=0.16,
    label="Prophet 80% interval",
)
ax.plot(
    validation_results["ds"],
    validation_results["weekday_mean_pred"],
    linestyle="--",
    label="Training weekday mean",
)
ax.set(
    title=f"Out-of-time validation · {selected_airport}",
    xlabel="Validation day",
    ylabel="Delayed arrivals (%)",
)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.2)
ax.legend(ncol=2)
plt.tight_layout()


## 6. Decision rule and limitations

Keep this experiment only if Prophet consistently improves validation MAE and weighted MAE over the weekday baseline. One airport and one validation month are not enough to claim a general improvement. The next stage would repeat the same frozen procedure across several high-volume airports and use rolling snapshot cutoffs.

Important limitations:

- The source contains non-consecutive monthly snapshots, so long-term trend estimates are weak.
- A percentage target is bounded, whereas standard Prophet uses an additive error model. Predictions are clipped only for evaluation.
- The daily rate has different precision depending on flight volume; weighted MAE partially reflects this.
- Do not feed validation outcomes, future airport volume, or future delay observations into training regressors.
- A useful deployment path is to create leakage-safe, out-of-fold airport forecasts and test them as an additional feature of the flight-level Ridge model.

## 7. Optional locked-period evaluation

Leave this disabled while making modelling decisions. Enable it only after the Prophet configuration and acceptance rule have been frozen using validation.

In [ ]:
EVALUATE_LOCKED_PERIODS = False

if EVALUATE_LOCKED_PERIODS:
    locked_frames = [
        build_daily_airport_series(PARTITIONS[name], selected_airport, name)
        for name in ("test", "future_test")
    ]
    locked_daily = pd.concat(locked_frames, ignore_index=True).sort_values("ds")
    frozen_training = pd.concat([train_daily, validation_daily], ignore_index=True)
    frozen_model = fit_prophet(frozen_training)
    locked_forecast = frozen_model.predict(locked_daily[["ds"]])
    locked_results = locked_daily.merge(
        locked_forecast[["ds", "yhat"]], on="ds", validate="one_to_one"
    )
    locked_results["prophet_pred"] = locked_results["yhat"].clip(0, 100)
    display(
        score_predictions(locked_results, {"Frozen Prophet weekly": "prophet_pred"})
    )
else:
    print("Locked March and June 2023 periods were not read.")
